In [1]:
from dotenv import load_dotenv

load_dotenv("/home/bart/.env.prod_wr")

from eyened_orm import Database, ImageInstance
db = Database()
session = db.create_session()
db.database_settings

user              : eyened_wr
password          : **********
host              : eyened-server
database          : eyened_database
port              : 22111
raise_on_warnings : True

In [2]:
from eyened_orm import StorageBackend

routing_settings = {"thumbnails": "thumbnails", "oogergo": "/mnt/oogergo", "genr": "/mnt/genr"}
for key, mount_point in routing_settings.items():
    storage_backend = StorageBackend.get_or_create(
        session,
        match_by={"Key": key},
        create_kwargs={"Key": key, "Kind": "local", "Config": {}},
    )
    session.add(storage_backend)
session.commit()
storage_backend_ids = {s.Key: s.StorageBackendID for s in StorageBackend.fetch_all(session)}
storage_backend_ids

{'thumbnails': 1, 'oogergo': 2, 'genr': 3}

In [3]:
all_images = ImageInstance.select(
    session, ImageInstance.ImageInstanceID, ImageInstance.DatasetIdentifier
)

In [4]:
from eyened_orm import ImageStorage

In [5]:
def get_storage(iid, dataset_identifier):
    def make(storage_id, object_key, fmt):
        return {
            "ImageInstanceID": iid,
            "StorageBackendID": storage_id,
            "ObjectKey": object_key,
            "Format": fmt,
            "IsPrimary": True,
        }

    if dataset_identifier.startswith("[png_series"):
        _, rest = dataset_identifier.split("]", 1)
        storage_key, object_key = rest.split("/", 1)
        return make(storage_backend_ids[storage_key], object_key, "png_series")

    suffix = dataset_identifier.rsplit(".", 1)[-1].lower()
    storage_key, path = dataset_identifier.split("/", 1)
    storage_id = storage_backend_ids[storage_key]

    if suffix == "json":
        return None
    if suffix == "binary":
        object_key = path.rsplit(".", 1)[0]
        return make(storage_id, object_key, "binary")
    if suffix == "dcm":
        return make(storage_id, path, "dicom")

    image_formats = {
        "png": "image/png",
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
    }
    fmt = image_formats.get(suffix)
    if fmt is None:
        raise ValueError(f"Unsupported suffix: {suffix!r}")

    return make(storage_id, path, fmt)

In [6]:
from itertools import islice
from tqdm import tqdm
def batched(iterable, size=5000):
    it = iter(iterable)
    while batch := list(islice(it, size)):
        yield batch


rows = (s for iid, di in all_images if (s := get_storage(iid, di)) is not None)

for batch in tqdm(batched(rows, 5000)):
    session.bulk_insert_mappings(ImageStorage, batch)
    session.commit()

367it [01:26,  4.25it/s]
